# ARM97 QMC64 RMSE / Correlation Visualization

This notebook implements Nathan's suggested visual diagnostics for `run_e3sm_scm_ARM97_qmc64_0706`:

1. RMSE versus correlation coefficient scatter plot for every QMC ensemble member, with the default-parameter model marked explicitly.
2. Time-series comparison for observation, default model, and all ensemble members, colored by performance relative to the default model.
3. A GeoCAT-style Taylor diagram that summarizes correlation, normalized standard deviation, centered RMS difference, and optional bias coloring.

The default paths below use the QMC64 output directory plus the existing ARM97 baseline model and observation files. Override them with environment variables if needed.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timedelta
import os
from pathlib import Path
import math

import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    return Path("/Users/yunlong/Workshop/SCM-UQ-Workflow")


ROOT = Path(os.environ.get("SCM_UQ_WORKFLOW_ROOT", find_repo_root())).resolve()
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".local_cache/matplotlib-cache"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".local_cache"))

CASE_DIR = Path(os.environ.get(
    "ARM97_QMC_CASE_DIR",
    str(ROOT / "cases/run_e3sm_scm_ARM97_qmc64_0706"),
)).expanduser().resolve()
QMC_OUTPUT_DIR = Path(os.environ.get(
    "ARM97_QMC_OUTPUT_DIR",
    str(CASE_DIR / "output"),
)).expanduser().resolve()
DEFAULT_MODEL_FILE = Path(os.environ.get(
    "ARM97_DEFAULT_MODEL_FILE",
    str(ROOT / "cases/run_e3sm_scm_ARM97_implicit_stress/arm97_implicit_stress_model_ready.nc"),
)).expanduser().resolve()
OBSERVATION_FILE = Path(os.environ.get(
    "ARM97_OBSERVATION_FILE",
    str(ROOT / "cases/run_e3sm_scm_ARM97_implicit_stress/arm97_iop_observation_model_window_nco.nc"),
)).expanduser().resolve()
OUT_DIR = Path(os.environ.get(
    "ARM97_QMC_VIS_OUT_DIR",
    str(CASE_DIR / "notebook_outputs/rmse_correlation_visualization"),
)).expanduser().resolve()

for path in [CASE_DIR, QMC_OUTPUT_DIR, DEFAULT_MODEL_FILE, OBSERVATION_FILE]:
    assert path.exists(), path
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 180,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.size": 10,
})

print("repo root:", ROOT)
print("case dir:", CASE_DIR)
print("QMC output dir:", QMC_OUTPUT_DIR)
print("default model:", DEFAULT_MODEL_FILE)
print("observation:", OBSERVATION_FILE)
print("output dir:", OUT_DIR)

## Variable Mapping

The mappings below follow the existing ARM97 observation-comparison notebooks. Surface variables are reduced over non-time dimensions. Profile variables are reduced over non-time, non-level dimensions and can be compared either at a selected observation pressure level or after averaging over selected pressure levels.

In [ ]:
@dataclass(frozen=True)
class VarSpec:
    model: str
    obs: str
    units: str
    scale_obs: float = 1.0
    obs_offset: float = 0.0
    description: str = ""


SURFACE_SPECS = {
    spec.model: spec
    for spec in [
        VarSpec("TREFHT", "Tsair", "K", description="2 m air temperature"),
        VarSpec("TS", "Tg", "K", description="surface / ground temperature"),
        VarSpec("TMQ", "prew", "kg m-2", scale_obs=10.0, description="precipitable water"),
        VarSpec("CLDTOT", "totcld", "1", scale_obs=0.01, description="total cloud fraction"),
        VarSpec("CLDLOW", "lowcld", "1", scale_obs=0.01, description="low cloud fraction"),
        VarSpec("CLDMED", "midcld", "1", scale_obs=0.01, description="mid-level cloud fraction"),
        VarSpec("CLDHGH", "hghcld", "1", scale_obs=0.01, description="high cloud fraction"),
        VarSpec("PS", "Ps", "Pa", description="surface pressure"),
        VarSpec("LHFLX", "lhflx", "W m-2", description="latent heat flux"),
        VarSpec("SHFLX", "shflx", "W m-2", description="sensible heat flux"),
        VarSpec("FSNS", "srfswdn-srfswup", "W m-2", description="surface net shortwave flux"),
        VarSpec("FLNS", "srflwup-srflwdn", "W m-2", description="surface net longwave flux"),
        VarSpec("FSDS", "srfswdn", "W m-2", description="surface downwelling shortwave flux"),
        VarSpec("FLDS", "srflwdn", "W m-2", description="surface downwelling longwave flux"),
        VarSpec("FLUT", "TOA_LWup", "W m-2", description="TOA upwelling longwave flux"),
        VarSpec("U10", "windsrf", "m s-1", description="10 m wind speed"),
        VarSpec("PRECT", "Prec", "m s-1", scale_obs=0.001, description="total precipitation rate"),
    ]
}

PROFILE_SPECS = {
    spec.model: spec
    for spec in [
        VarSpec("T", "T", "K", description="temperature"),
        VarSpec("Q", "q", "kg kg-1", description="specific humidity"),
        VarSpec("U", "u", "m s-1", description="zonal wind"),
        VarSpec("V", "v", "m s-1", description="meridional wind"),
        VarSpec("OMEGA", "omega", "Pa s-1", description="pressure vertical velocity"),
        VarSpec("RELHUM", "rh", "%", description="relative humidity"),
    ]
}

# Primary diagnostic variable. Change this cell and re-run subsequent cells for another variable.
VARIABLE = "PRECT"

# For profile variables, choose one pressure level in hPa, or set PROFILE_LEVEL_HPA = None
# and use PROFILE_LAYER_HPA to average over a pressure range.
PROFILE_LEVEL_HPA = 500.0
PROFILE_LAYER_HPA = None  # example: (300.0, 900.0)

# Time-window crop for all metrics and plots. Leave as None to use the overlap of model and observation.
START_TIME = None  # example: "1997-06-25"
END_TIME = None    # example: "1997-07-05"

# Visual threshold for neutral time-series coloring. Members within this fraction of default RMSE
# are drawn with a neutral color; lower RMSE is blue and higher RMSE is red.
NEUTRAL_RMSE_FRAC = 0.02

## Data Loading and Metric Functions

In [ ]:
def filled(var_or_array):
    return np.asarray(np.ma.asarray(var_or_array, dtype=np.float64).filled(np.nan), dtype=np.float64)


def reduce_to_time_series(values, dims):
    values = filled(values)
    axes = tuple(i for i, dim in enumerate(dims) if dim != "time")
    if axes:
        return np.nanmean(values, axis=axes)
    return values


def load_model_days_dates(ds):
    time = ds.variables["time"]
    days = np.asarray(time[:], dtype=np.float64)
    dates = np.array(
        num2date(days, time.units, getattr(time, "calendar", "standard"), only_use_cftime_datetimes=False),
        dtype=object,
    )
    return days, dates


def parse_bdate(ds):
    value = int(np.asarray(ds.variables["bdate"][...]).item())
    text = str(value)
    if len(text) == 8:
        return datetime(int(text[:4]), int(text[4:6]), int(text[6:8]))
    if len(text) == 6:
        year = int(text[:2])
        year += 1900 if year >= 70 else 2000
        return datetime(year, int(text[2:4]), int(text[4:6]))
    raise ValueError(f"unsupported bdate value: {value}")


def load_obs_dates(ds):
    if "bdate" in ds.variables and "tsec" in ds.variables:
        base = parse_bdate(ds)
        tsec = np.asarray(ds.variables["tsec"][:], dtype=np.float64)
        return np.array([base + timedelta(seconds=float(x)) for x in tsec], dtype=object)
    time = ds.variables["time"]
    return np.array(
        num2date(time[:], time.units, getattr(time, "calendar", "standard"), only_use_cftime_datetimes=False),
        dtype=object,
    )


def obs_days_from_model_origin(obs_dates, model_origin):
    return np.asarray([(d - model_origin).total_seconds() / 86400.0 for d in obs_dates], dtype=np.float64)


def obs_names_required(expression):
    return [name.strip() for name in expression.replace("-", ",").split(",") if name.strip()]


def obs_expr_series(ds, expression):
    parts = [part.strip() for part in expression.split("-")]
    values = reduce_to_time_series(ds.variables[parts[0]][:], tuple(ds.variables[parts[0]].dimensions))
    for part in parts[1:]:
        values = values - reduce_to_time_series(ds.variables[part][:], tuple(ds.variables[part].dimensions))
    return values


def vertical_dim(dims):
    for dim in ("lev", "ilev"):
        if dim in dims:
            return dim
    return None


def level_values_hpa(ds, dim_name):
    levels = filled(ds.variables[dim_name][:])
    units = getattr(ds.variables[dim_name], "units", "").lower()
    if "pa" in units and "hpa" not in units:
        levels = levels / 100.0
    return levels


def profile_to_pressure_series(ds, variable_name, target_hpa=None, layer_hpa=None):
    var = ds.variables[variable_name]
    dims = tuple(var.dimensions)
    lev_dim = vertical_dim(dims)
    if lev_dim is None:
        raise ValueError(f"{variable_name} has no lev/ilev dimension")
    values = filled(var[:])
    lev_axis = dims.index(lev_dim)
    levels = level_values_hpa(ds, lev_dim)

    # Reduce dimensions other than time and the vertical level.
    reduce_axes = tuple(i for i, dim in enumerate(dims) if dim not in {"time", lev_dim})
    if reduce_axes:
        values = np.nanmean(values, axis=reduce_axes)
        remaining_dims = [dim for dim in dims if dim not in {dims[i] for i in reduce_axes}]
        lev_axis = remaining_dims.index(lev_dim)
    if lev_axis != 1:
        values = np.moveaxis(values, lev_axis, 1)

    if layer_hpa is not None:
        lo, hi = sorted(layer_hpa)
        mask = (levels >= lo) & (levels <= hi)
        if not np.any(mask):
            raise ValueError(f"no levels found inside {layer_hpa} hPa")
        return np.nanmean(values[:, mask], axis=1), f"{lo:g}-{hi:g} hPa mean"

    if target_hpa is None:
        target_hpa = float(levels[np.nanargmin(np.abs(levels - 500.0))])
    idx = int(np.nanargmin(np.abs(levels - target_hpa)))
    return values[:, idx], f"{levels[idx]:g} hPa"


def model_series(model_file, variable_name, profile_level_hpa=None, profile_layer_hpa=None):
    with Dataset(model_file) as ds:
        days, dates = load_model_days_dates(ds)
        if variable_name in PROFILE_SPECS and vertical_dim(ds.variables[variable_name].dimensions) is not None:
            values, level_label = profile_to_pressure_series(ds, variable_name, profile_level_hpa, profile_layer_hpa)
        else:
            var = ds.variables[variable_name]
            values = reduce_to_time_series(var[:], tuple(var.dimensions))
            level_label = ""
    return days, dates, np.asarray(values, dtype=np.float64), level_label


def observation_series(obs_file, model_origin, variable_name, profile_level_hpa=None, profile_layer_hpa=None):
    spec = PROFILE_SPECS.get(variable_name) or SURFACE_SPECS.get(variable_name)
    if spec is None:
        raise KeyError(f"No observation mapping configured for {variable_name}")
    with Dataset(obs_file) as ds:
        missing = [name for name in obs_names_required(spec.obs) if name not in ds.variables]
        if missing:
            raise KeyError(f"Observation file is missing {missing} for {variable_name}")
        obs_dates = load_obs_dates(ds)
        obs_days = obs_days_from_model_origin(obs_dates, model_origin)
        if variable_name in PROFILE_SPECS:
            values, level_label = profile_to_pressure_series(ds, spec.obs, profile_level_hpa, profile_layer_hpa)
        else:
            values = obs_expr_series(ds, spec.obs)
            level_label = ""
    values = np.asarray(values, dtype=np.float64) * spec.scale_obs + spec.obs_offset
    return obs_days, obs_dates, values, spec, level_label


def interpolate_to_model(obs_days, obs_values, target_days):
    finite = np.isfinite(obs_days) & np.isfinite(obs_values)
    if finite.sum() < 2:
        return np.full_like(target_days, np.nan, dtype=np.float64)
    order = np.argsort(obs_days[finite])
    return np.interp(target_days, obs_days[finite][order], obs_values[finite][order], left=np.nan, right=np.nan)


def time_window_mask(dates, start_time=None, end_time=None):
    stamps = pd.to_datetime([str(x) for x in dates], format="mixed")
    mask = np.ones(len(stamps), dtype=bool)
    if start_time is not None:
        mask &= stamps >= pd.Timestamp(start_time)
    if end_time is not None:
        mask &= stamps <= pd.Timestamp(end_time)
    return mask


def rmse(model, obs):
    mask = np.isfinite(model) & np.isfinite(obs)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.nanmean((model[mask] - obs[mask]) ** 2)))


def corrcoef(model, obs):
    mask = np.isfinite(model) & np.isfinite(obs)
    if mask.sum() < 3 or np.nanstd(model[mask]) == 0 or np.nanstd(obs[mask]) == 0:
        return np.nan
    return float(np.corrcoef(model[mask], obs[mask])[0, 1])


def centered_rms(model, obs):
    mask = np.isfinite(model) & np.isfinite(obs)
    if mask.sum() == 0:
        return np.nan
    m = model[mask] - np.nanmean(model[mask])
    o = obs[mask] - np.nanmean(obs[mask])
    return float(np.sqrt(np.nanmean((m - o) ** 2)))


def std_ratio(model, obs):
    mask = np.isfinite(model) & np.isfinite(obs)
    if mask.sum() < 2 or np.nanstd(obs[mask]) == 0:
        return np.nan
    return float(np.nanstd(model[mask]) / np.nanstd(obs[mask]))


def qmc_member_files(output_dir=QMC_OUTPUT_DIR):
    return sorted(output_dir.glob("ARM97_qmc_*.nc"))


def member_label(path):
    return path.stem.replace("ARM97_qmc_", "qmc_")

## Compute Metrics for Default and QMC64 Members

Each row below compares a model time series to the observation interpolated onto that model's time axis. `delta_rmse_vs_default < 0` means the member has lower RMSE than the default-parameter model. `delta_corr_vs_default > 0` means higher correlation than default.

In [ ]:
def build_metric_table(variable_name=VARIABLE):
    default_days, default_dates, default_values, default_level_label = model_series(
        DEFAULT_MODEL_FILE, variable_name, PROFILE_LEVEL_HPA, PROFILE_LAYER_HPA
    )
    obs_days, obs_dates, obs_values, spec, obs_level_label = observation_series(
        OBSERVATION_FILE,
        default_dates[0],
        variable_name,
        PROFILE_LEVEL_HPA,
        PROFILE_LAYER_HPA,
    )
    obs_on_default = interpolate_to_model(obs_days, obs_values, default_days)
    default_window = time_window_mask(default_dates, START_TIME, END_TIME)

    rows = []
    def add_row(label, kind, sample_index, path, days, dates, values):
        obs_on_model = interpolate_to_model(obs_days, obs_values, days)
        mask = time_window_mask(dates, START_TIME, END_TIME)
        values_w = values[mask]
        obs_w = obs_on_model[mask]
        rows.append({
            "label": label,
            "kind": kind,
            "sample_index": sample_index,
            "file": str(path),
            "n_valid": int(np.sum(np.isfinite(values_w) & np.isfinite(obs_w))),
            "rmse": rmse(values_w, obs_w),
            "correlation": corrcoef(values_w, obs_w),
            "bias": float(np.nanmean(values_w - obs_w)),
            "model_std": float(np.nanstd(values_w)),
            "obs_std": float(np.nanstd(obs_w)),
            "std_ratio": std_ratio(values_w, obs_w),
            "centered_rms": centered_rms(values_w, obs_w),
        })
        return obs_on_model

    add_row("default", "default", -1, DEFAULT_MODEL_FILE, default_days, default_dates, default_values)

    series = {
        "default": {
            "dates": default_dates,
            "days": default_days,
            "values": default_values,
            "obs_on_model": obs_on_default,
            "file": DEFAULT_MODEL_FILE,
        }
    }

    for path in qmc_member_files():
        days, dates, values, _ = model_series(path, variable_name, PROFILE_LEVEL_HPA, PROFILE_LAYER_HPA)
        label = member_label(path)
        sample_index = int(label.split("_")[-1])
        obs_on_model = add_row(label, "qmc", sample_index, path, days, dates, values)
        series[label] = {"dates": dates, "days": days, "values": values, "obs_on_model": obs_on_model, "file": path}

    table = pd.DataFrame(rows)
    default_rmse = float(table.loc[table["kind"] == "default", "rmse"].iloc[0])
    default_corr = float(table.loc[table["kind"] == "default", "correlation"].iloc[0])
    table["delta_rmse_vs_default"] = table["rmse"] - default_rmse
    table["delta_corr_vs_default"] = table["correlation"] - default_corr
    table["skill_rmse_vs_default_pct"] = 100.0 * (default_rmse - table["rmse"]) / default_rmse
    table = table.sort_values(["kind", "rmse"], ascending=[True, True]).reset_index(drop=True)

    metadata = {
        "variable": variable_name,
        "description": spec.description,
        "units": spec.units,
        "obs_mapping": spec.obs,
        "default_level_label": default_level_label,
        "obs_level_label": obs_level_label,
        "obs_days": obs_days,
        "obs_dates": obs_dates,
        "obs_values": obs_values,
    }
    return table, series, metadata


METRICS, SERIES, META = build_metric_table(VARIABLE)
metrics_out = OUT_DIR / f"{VARIABLE}_rmse_correlation_metrics.csv"
METRICS.to_csv(metrics_out, index=False)
print("wrote", metrics_out)
print(META)
METRICS.head(12)

## 1. RMSE vs Correlation Scatter

The lower-right region is better: lower RMSE and higher correlation. The dashed lines mark the default-parameter model.

In [ ]:
def plot_rmse_correlation(metrics=METRICS, variable_name=VARIABLE, meta=META):
    default = metrics[metrics["kind"] == "default"].iloc[0]
    qmc = metrics[metrics["kind"] == "qmc"].copy()

    fig, ax = plt.subplots(figsize=(8.6, 6.1))
    colors = np.where(
        (qmc["rmse"] < default["rmse"]) & (qmc["correlation"] > default["correlation"]),
        "#1f77b4",
        np.where(qmc["rmse"] < default["rmse"], "#2ca02c", np.where(qmc["correlation"] > default["correlation"], "#9467bd", "#d95f02")),
    )
    ax.scatter(qmc["rmse"], qmc["correlation"], c=colors, s=52, alpha=0.82, edgecolor="white", linewidth=0.5)
    ax.scatter(default["rmse"], default["correlation"], marker="*", s=260, c="black", edgecolor="white", linewidth=0.8, zorder=5, label="default")
    ax.axvline(default["rmse"], color="black", ls="--", lw=1.0, alpha=0.55)
    ax.axhline(default["correlation"], color="black", ls="--", lw=1.0, alpha=0.55)

    best = qmc.sort_values("rmse").head(5)
    for _, row in best.iterrows():
        ax.annotate(str(int(row["sample_index"])), (row["rmse"], row["correlation"]), xytext=(4, 4), textcoords="offset points", fontsize=8)

    ax.set_xlabel(f"RMSE ({meta['units']})")
    ax.set_ylabel("Correlation coefficient")
    title = f"{variable_name}: RMSE vs correlation"
    if meta.get("default_level_label") or meta.get("obs_level_label"):
        title += f" ({meta.get('default_level_label') or meta.get('obs_level_label')})"
    ax.set_title(title)

    legend_items = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#1f77b4", label="lower RMSE and higher corr", markersize=8),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#2ca02c", label="lower RMSE only", markersize=8),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#9467bd", label="higher corr only", markersize=8),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="#d95f02", label="worse or mixed", markersize=8),
        Line2D([0], [0], marker="*", color="w", markerfacecolor="black", label="default", markersize=13),
    ]
    ax.legend(handles=legend_items, frameon=False, loc="best")
    fig.tight_layout()
    out = OUT_DIR / f"{variable_name}_rmse_vs_correlation.png"
    fig.savefig(out, bbox_inches="tight")
    print("wrote", out)
    return fig

plot_rmse_correlation();

## 2. Time-Series Comparison Colored by Relative Performance

Blue members have lower RMSE than the default model, red members have higher RMSE, and gray members are within `NEUTRAL_RMSE_FRAC` of the default RMSE. The observation and default model are drawn on top.

In [ ]:
def classify_member(row, default_rmse):
    delta = row["rmse"] - default_rmse
    if abs(delta) <= NEUTRAL_RMSE_FRAC * default_rmse:
        return "near_default"
    return "better" if delta < 0 else "worse"


def plot_colored_time_series(metrics=METRICS, series=SERIES, variable_name=VARIABLE, meta=META):
    default_row = metrics[metrics["kind"] == "default"].iloc[0]
    qmc_rows = metrics[metrics["kind"] == "qmc"].copy()
    qmc_rows["class"] = qmc_rows.apply(lambda row: classify_member(row, default_row["rmse"]), axis=1)

    palette = {"better": "#1f77b4", "near_default": "#8c8c8c", "worse": "#c44e52"}
    alphas = {"better": 0.42, "near_default": 0.28, "worse": 0.30}

    fig, ax = plt.subplots(figsize=(12.0, 5.8))
    for cls in ["worse", "near_default", "better"]:
        subset = qmc_rows[qmc_rows["class"] == cls]
        for _, row in subset.iterrows():
            item = series[row["label"]]
            mask = time_window_mask(item["dates"], START_TIME, END_TIME)
            ax.plot(item["dates"][mask], item["values"][mask], color=palette[cls], alpha=alphas[cls], lw=0.9)

    default_item = series["default"]
    mask = time_window_mask(default_item["dates"], START_TIME, END_TIME)
    ax.plot(default_item["dates"][mask], default_item["values"][mask], color="black", lw=2.0, label="default")
    ax.plot(default_item["dates"][mask], default_item["obs_on_model"][mask], color="#f0b400", lw=2.4, label=f"observation: {meta['obs_mapping']}")

    counts = qmc_rows["class"].value_counts().to_dict()
    handles = [
        Line2D([0], [0], color="#f0b400", lw=2.4, label="observation"),
        Line2D([0], [0], color="black", lw=2.0, label="default"),
        Line2D([0], [0], color=palette["better"], lw=2, label=f"better RMSE ({counts.get('better', 0)})"),
        Line2D([0], [0], color=palette["near_default"], lw=2, label=f"near default ({counts.get('near_default', 0)})"),
        Line2D([0], [0], color=palette["worse"], lw=2, label=f"worse RMSE ({counts.get('worse', 0)})"),
    ]
    ax.legend(handles=handles, frameon=False, ncol=3, loc="upper left")
    ax.set_xlabel("Time")
    ax.set_ylabel(f"{variable_name} ({meta['units']})")
    title = f"{variable_name}: observation, default, and QMC64 members"
    if meta.get("default_level_label") or meta.get("obs_level_label"):
        title += f" ({meta.get('default_level_label') or meta.get('obs_level_label')})"
    ax.set_title(title)
    fig.autofmt_xdate()
    fig.tight_layout()
    out = OUT_DIR / f"{variable_name}_colored_time_series.png"
    fig.savefig(out, bbox_inches="tight")
    print("wrote", out)
    return fig

plot_colored_time_series();

## 3. GeoCAT-style Taylor Diagram

This follows the layout used by `geocat.viz.taylor.TaylorDiagram`: normalized standard deviation is the radial coordinate, correlation is shown on the curved angular axis, the observation reference is at `(corr=1, std=1)`, and dashed contours show centered RMS difference. The implementation below is pure Matplotlib so the notebook still runs when `geocat-viz` is not installed. API reference: https://geocat-viz.readthedocs.io/en/latest/user_api/generated/geocat.viz.taylor.TaylorDiagram.html

In [ ]:
def _nice_std_levels(max_radius):
    if max_radius <= 1.65:
        return np.array([0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5])
    step = 0.25 if max_radius <= 2.0 else 0.5
    return np.arange(0.0, math.ceil(max_radius / step) * step + 0.5 * step, step)


def _corr_ticks_for_taylor(data):
    min_corr = float(np.nanmin(data["correlation"]))
    if min_corr >= 0.0:
        return np.array([1.0, 0.99, 0.95, 0.9, 0.8, 0.7, 0.6, 0.4, 0.2, 0.0])
    return np.array([1.0, 0.99, 0.95, 0.9, 0.8, 0.6, 0.4, 0.2, 0.0, -0.2, -0.5, -1.0])


def plot_geocat_style_taylor(metrics=METRICS, variable_name=VARIABLE, annotate_best=5, color_by="bias"):
    data = metrics[np.isfinite(metrics["correlation"]) & np.isfinite(metrics["std_ratio"])].copy()
    data["theta"] = np.arccos(np.clip(data["correlation"], -1.0, 1.0))
    qmc = data[data["kind"] == "qmc"].copy()
    default = data[data["kind"] == "default"].iloc[0]

    theta_max = float(np.nanmax(data["theta"]))
    theta_max = max(theta_max, np.pi / 2)
    theta_max = min(np.pi, theta_max * 1.04)
    max_radius = max(1.65, float(np.nanmax(data["std_ratio"])) * 1.10)
    std_levels = _nice_std_levels(max_radius)
    corr_ticks = _corr_ticks_for_taylor(data)
    corr_ticks = corr_ticks[np.arccos(np.clip(corr_ticks, -1.0, 1.0)) <= theta_max + 1e-9]

    fig = plt.figure(figsize=(10.0, 9.0))
    ax = fig.add_subplot(111, projection="polar")
    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)
    ax.set_thetamin(0)
    ax.set_thetamax(np.degrees(theta_max))
    ax.set_ylim(0, max_radius)
    ax.set_rgrids(std_levels[std_levels > 0], angle=135 if theta_max > np.pi / 2 else 100)
    ax.set_thetagrids(np.degrees(np.arccos(np.clip(corr_ticks, -1.0, 1.0))), labels=[f"{c:g}" for c in corr_ticks])
    ax.tick_params(axis="both", labelsize=10)
    ax.grid(color="0.82", linestyle=(0, (9, 5)), linewidth=0.7)

    # GeoCAT-style correlation and standard-deviation guide lines.
    for corr in corr_ticks:
        theta = math.acos(float(np.clip(corr, -1.0, 1.0)))
        ax.plot([theta, theta], [0, max_radius], color="0.88", linestyle=(0, (9, 5)), linewidth=0.55, zorder=0)
    theta_grid = np.linspace(0.0, theta_max, 361)
    for radius in std_levels[std_levels > 0]:
        ax.plot(theta_grid, np.full_like(theta_grid, radius), color="0.88", linestyle=(0, (9, 5)), linewidth=0.65, zorder=0)

    # Centered RMS difference contours normalized by observation standard deviation.
    theta = np.linspace(0.0, theta_max, 361)
    radius = np.linspace(0.0, max_radius, 260)
    T, R = np.meshgrid(theta, radius)
    centered = np.sqrt(np.maximum(0.0, R**2 + 1.0 - 2.0 * R * np.cos(T)))
    contour_levels = np.array([0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0])
    contour_levels = contour_levels[contour_levels < np.nanmax(centered)]
    contours = ax.contour(T, R, centered, levels=contour_levels, colors="0.48", linestyles="dashed", linewidths=0.9)
    ax.clabel(contours, inline=True, fontsize=8, fmt="%.2g")

    # Observation reference point and reference standard deviation arc.
    ax.scatter(0.0, 1.0, marker="o", s=110, c="#f0b400", edgecolor="black", linewidth=0.7, label="Observation", zorder=6)
    ax.plot(theta_grid, np.ones_like(theta_grid), color="#f0b400", linewidth=1.2, alpha=0.75, zorder=1)

    if color_by == "bias":
        color_values = qmc["bias"].to_numpy(dtype=float)
        vmax = np.nanmax(np.abs(color_values))
        norm = mpl.colors.TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax) if np.isfinite(vmax) and vmax > 0 else None
        sc = ax.scatter(qmc["theta"], qmc["std_ratio"], c=color_values, cmap="coolwarm", norm=norm,
                        s=62, alpha=0.86, edgecolor="white", linewidth=0.45, label="QMC members", zorder=4)
        cb_label = "Bias (model - obs)"
    else:
        color_values = qmc["rmse"].to_numpy(dtype=float)
        sc = ax.scatter(qmc["theta"], qmc["std_ratio"], c=color_values, cmap="viridis_r",
                        s=62, alpha=0.86, edgecolor="white", linewidth=0.45, label="QMC members", zorder=4)
        cb_label = "RMSE"

    ax.scatter(default["theta"], default["std_ratio"], marker="*", s=260, c="black",
               edgecolor="white", linewidth=0.8, label="Default", zorder=7)

    best = qmc.sort_values("rmse").head(int(annotate_best))
    for _, row in best.iterrows():
        ax.annotate(str(int(row["sample_index"])), (row["theta"], row["std_ratio"]),
                    xytext=(5, 5), textcoords="offset points", fontsize=8, zorder=8)

    cb = fig.colorbar(sc, ax=ax, pad=0.10, shrink=0.78)
    cb.set_label(cb_label)
    ax.set_title(f"{variable_name}: GeoCAT-style Taylor diagram", fontsize=16, y=1.08)
    ax.text(0.5, -0.10, "Correlation coefficient", transform=ax.transAxes, ha="center", va="center", fontsize=12)
    ax.text(-0.08, 0.45, "Normalized standard deviation", transform=ax.transAxes,
            ha="center", va="center", rotation=90, fontsize=12)
    ax.text(0.70, 0.84, "Centered RMS difference", transform=ax.transAxes, fontsize=9, color="0.35")
    ax.legend(frameon=False, loc="upper right", bbox_to_anchor=(1.23, 1.05))
    fig.tight_layout()
    out = OUT_DIR / f"{variable_name}_geocat_style_taylor.png"
    fig.savefig(out, bbox_inches="tight")
    print("wrote", out)
    return fig


# Backward-compatible alias for older cells or external use.
def plot_taylor_like(metrics=METRICS, variable_name=VARIABLE):
    return plot_geocat_style_taylor(metrics, variable_name)


plot_geocat_style_taylor();

## Best Members and Trade-Offs

Use this table to identify members that beat the default in RMSE, correlation, or both.

In [ ]:
def summarize_tradeoffs(metrics=METRICS):
    default = metrics[metrics["kind"] == "default"].iloc[0]
    qmc = metrics[metrics["kind"] == "qmc"].copy()
    qmc["lower_rmse_than_default"] = qmc["rmse"] < default["rmse"]
    qmc["higher_corr_than_default"] = qmc["correlation"] > default["correlation"]
    qmc["tradeoff"] = np.select(
        [
            qmc["lower_rmse_than_default"] & qmc["higher_corr_than_default"],
            qmc["lower_rmse_than_default"] & ~qmc["higher_corr_than_default"],
            ~qmc["lower_rmse_than_default"] & qmc["higher_corr_than_default"],
        ],
        ["better_rmse_and_corr", "lower_rmse_only", "higher_corr_only"],
        default="not_better_than_default",
    )
    cols = [
        "sample_index", "label", "rmse", "correlation", "bias", "std_ratio",
        "delta_rmse_vs_default", "delta_corr_vs_default", "skill_rmse_vs_default_pct", "tradeoff", "file",
    ]
    summary = qmc[cols].sort_values(["tradeoff", "rmse", "correlation"], ascending=[True, True, False])
    out = OUT_DIR / f"{VARIABLE}_tradeoff_summary.csv"
    summary.to_csv(out, index=False)
    print("default:")
    display(default[["rmse", "correlation", "bias", "std_ratio"]].to_frame().T)
    print("wrote", out)
    return summary

TRADEOFFS = summarize_tradeoffs()
TRADEOFFS.head(20)

## Batch Run for Several Variables

Optionally run this cell to generate the same three figures and metric CSVs for multiple mapped variables. For profile variables, it uses the level/layer settings from the configuration cell.

In [ ]:
BATCH_VARIABLES = ["TREFHT", "TMQ", "CLDTOT", "PRECT", "LHFLX", "SHFLX", "FSNS", "FLNS", "T", "Q"]


def run_batch(variables=BATCH_VARIABLES):
    outputs = []
    for variable in variables:
        try:
            metrics, series, meta = build_metric_table(variable)
            metrics.to_csv(OUT_DIR / f"{variable}_rmse_correlation_metrics.csv", index=False)
            plot_rmse_correlation(metrics, variable, meta)
            plot_colored_time_series(metrics, series, variable, meta)
            plot_geocat_style_taylor(metrics, variable)
            outputs.append({"variable": variable, "status": "ok"})
            plt.close("all")
        except Exception as exc:
            outputs.append({"variable": variable, "status": f"failed: {exc}"})
    return pd.DataFrame(outputs)

# Uncomment to run the batch after reviewing the single-variable output.
# run_batch()